# Tree-model experiments — with random & spatial cross-validation

This notebook trains DecisionTree / RandomForest / LGBM models to predict
soil properties, evaluated under several cross-validation strategies:

- **random** — plain random k-fold CV (`randomCV.py`)
- **block** — grid-based spatial block CV (`spatialCV.py: block_cv`)
- **sp1** — AHC spatial blocks CV (`spatialCV.py: sp1_cv`)

For every strategy the fold assignment is generated **once**, plotted, and
saved to disk (`./plots/cv_folds/<method>/...png` + `.csv`) before any model
is trained, then reused identically for every model / soil property /
normalization combination so all models are judged on the exact same
splits.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np
import os
os.environ['USE_PYGEOS'] = '0'
from itertools import product
import math
import time

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from joblib import dump

from Models.trees import *

# Cross-validation splitting utilities (shared between random & spatial CV)
from CVs.randomCV import random_cv, rdm_cv_split
from CVs.spatialCV import block_cv, sp1_cv, sp_cv, sp2_cv
from CVs.cv_plot_utils import plot_folds
import geopandas as gpd 

In [3]:
import os
import multiprocessing
import sys
# Set CPU threads BEFORE importing TensorFlow
num_cores = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(num_cores)
os.environ["TF_NUM_INTRAOP_THREADS"] = str(num_cores)
os.environ["TF_NUM_INTEROP_THREADS"] = str(num_cores)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0 --tf_xla_enable_xla_devices=false'
sys.stderr = open(os.devnull, 'w')

In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[]


## Load dataset

In [6]:
dataframe = pd.read_csv("../Data/unitask_data/dataset.csv")

# randomCV.py / spatialCV.py both key everything off a stable point ID
# (points[:, 0] = ID). Make sure one exists.
# dataframe = dataframe.reset_index(drop=True)
# if 'ID' not in dataframe.columns:
#     dataframe['ID'] = dataframe.index

# dataframe.head()

## Useful functions

In [7]:
def create_results_csv(csv_columns, csv_name):
    experiments = csv_name

    if os.path.exists(f"../Results/{experiments}"):
        df_experiments = pd.read_csv(f"../Results/{experiments}")
    else:
        df_experiments = pd.DataFrame(columns=csv_columns)
        df_experiments.to_csv(f"../Results/{experiments}", index=False)

    return df_experiments

In [8]:
# Determining features and target columns
columns = dataframe.columns.to_list()
soil_properties = ['pH', 'CE', 'CEC', 'OM', 'A%', 'L%', 'S%']
cols_to_drop = soil_properties + ['X', 'Y']
covariates = [col for col in columns if col not in cols_to_drop]

In [9]:
gdf = gpd.GeoDataFrame(
    dataframe,
    geometry=gpd.points_from_xy(dataframe['X'], dataframe['Y']),
    crs='EPSG:4326'
)

gdf = gdf.to_crs('EPSG:32631')

# Keep expected 47-column layout: covariates + soil properties + X + Y
ordered_cols = covariates + soil_properties + ['X', 'Y']

dataframe_proj = dataframe.copy()
dataframe_proj['X'] = gdf.geometry.x.values
dataframe_proj['Y'] = gdf.geometry.y.values

points = dataframe_proj[ordered_cols].to_numpy()
print("Points shape:", points.shape)

Points shape: (2156, 47)


In [10]:
def check_existance_params_in_experiments(df_experiments, config_to_check):

    # Extract relevant columns from DataFrame
    relevant_columns = list(config_to_check.keys())
    relevant_df = df_experiments[relevant_columns]
    # Ensure data types match
    for col in relevant_columns:
        if col in relevant_df and col in config_to_check:
            # Convert lists to strings for comparison
            if config_to_check[col] is not None:
                if len(relevant_df[col]) > 0 and isinstance(relevant_df[col].iloc[0], list):
                    relevant_df[col] = relevant_df[col].apply(str)
                    config_to_check[col] = str(config_to_check[col])
                # Otherwise, cast to DataFrame's column type
                elif relevant_df[col].dtype == 'object':  # Handle object (or string-like) columns
                    config_to_check[col] = str(config_to_check[col])
                else:
                    try:
                        config_to_check[col] = relevant_df[col].dtype.type(config_to_check[col])
                    except ValueError:
                        # keep the original (string) value
                        pass
            else:
                config_to_check[col] = ""
                relevant_df[col].fillna("", inplace=True)

    # Convert the configuration to a DataFrame row
    config_df = pd.DataFrame([config_to_check])

    # Check if the configuration exists
    exists = any((relevant_df == config_df.iloc[0]).all(axis=1))
    return exists

## Cross-validation folds



In [11]:
from CVs.cv_utils import build_folds_by_method

N_FOLDS = 5
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


block_thresholds_km = [5, 10, 20, 30, 50, 100, 150, 200]
sp1_thresholds_km = [5, 10, 20, 30, 50, 100, 150, 200]
sp2_thresholds_km = [5, 10, 20, 30, 50, 100, 150, 200]


block_thresholds_m = [v * 1000 for v in block_thresholds_km]
sp1_thresholds_m = [v * 1000 for v in sp1_thresholds_km]
sp2_thresholds_m = [v * 1000 for v in sp2_thresholds_km]

n_cov = len(covariates)        
n_tar = len(soil_properties) 
print(n_cov, n_tar) 

env = points[:, :n_cov]                
tar = points[:, n_cov:n_cov + n_tar]  
REGION = 'all'
folds_by_method = build_folds_by_method(
    region = REGION,
    points = points,
    env = env,
    tar = tar,
    n_folds=N_FOLDS,
    block_sp_threshold=block_thresholds_m,
    sp1_threshold=sp1_thresholds_m,
    sp_threshold=sp2_thresholds_m,
    random_repeats=1
)


print(sorted(folds_by_method.keys()))

for name, folds in folds_by_method.items():
    n_pts = len(folds)
    n_f = len(set(int(row[1]) for row in folds))
    print(f"{name:>6}: {n_pts} points assigned across {n_f} folds")

38 7
d:\DSM_MODULE\CVs\cv_folds\all\random
d:\DSM_MODULE\CVs\cv_folds\all\sp1
d:\DSM_MODULE\CVs\cv_folds\all\sp2
d:\DSM_MODULE\CVs\cv_folds\all\sp
Building folds by method in d:\DSM_MODULE\CVs\cv_folds\all...
  - Random CV: 1 repeats, k=5
Generating random-CV folds.
Checking for existing CSV: d:\DSM_MODULE\CVs\cv_folds\all\random\randomCV_rep0_k5.csv
  - Block CV: k=5, thresholds=[5000, 10000, 20000, 30000, 50000, 100000, 150000, 200000]
    - Threshold: 5000
Generating block-CV folds.
    - Threshold: 10000
Generating block-CV folds.
    - Threshold: 20000
Generating block-CV folds.
    - Threshold: 30000
Generating block-CV folds.
    - Threshold: 50000
Generating block-CV folds.
    - Threshold: 100000
Generating block-CV folds.
    - Threshold: 150000
Generating block-CV folds.
    - Threshold: 200000
Generating block-CV folds.
  - SP1 CV: k=5, thresholds=[5000, 10000, 20000, 30000, 50000, 100000, 150000, 200000]
    - Threshold: 5000
    - Threshold: 10000
    - Threshold: 20000
 

In [12]:
folds_by_method.keys()

dict_keys(['random', 'block_k5_5000', 'block_k5_10000', 'block_k5_20000', 'block_k5_30000', 'block_k5_50000', 'block_k5_100000', 'block_k5_150000', 'block_k5_200000', 'sp1_k5_5000', 'sp1_k5_10000', 'sp1_k5_20000', 'sp1_k5_30000', 'sp1_k5_50000', 'sp1_k5_100000', 'sp1_k5_150000', 'sp1_k5_200000', 'sp2_k5', 'sp_k5_5000', 'sp_k5_10000', 'sp_k5_20000', 'sp_k5_30000', 'sp_k5_50000', 'sp_k5_100000', 'sp_k5_150000', 'sp_k5_200000'])

### (Optional) target-aware spatial CV — `sp_cv` / `sp2_cv`

`spatialCV.sp_cv` / `spatialCV.sp2_cv` additionally cluster on the *target*
values, so — unlike `random`/`block`/`sp1` above — their folds are specific
to a single soil property and must be regenerated per property. They also
need the optional `kmodes` and `ClusterEnsembles` packages. They're not part
of the default sweep below (much more expensive: re-clustering per soil
property), but are wired up and ready to use:

```python
s_property = 'pH'
env = dataframe[['ID'] + covariates].to_numpy(dtype=float)
tar = dataframe[['ID', s_property]].to_numpy(dtype=float)

# cate_col_start=0, cate_col_end=-1 => no categorical covariates
sp_folds = sp_cv(
    points, env, tar, cate_col_start=0, cate_col_end=-1,
    sp_threshold=BLOCK_SP_THRESHOLD, k=N_FOLDS,
    out_dir=os.path.join(CV_PLOTS_DIR, 'sp', s_property), basename='spCV'
)
```

## Main training and validation function

In [13]:
def build_group_array(dataframe, folds, id_col='ID'):
    """Map each row of `dataframe` to its assigned fold.

    The CV functions use the dataframe row index as the point ID because
    the dataset has no explicit ID column.

    `folds` contains [point_ID, fold_index, ...].
    """

    fold_map = {
        int(row[0]): int(row[1])
        for row in folds
    }

    # No explicit ID column in the dataset.
    # The row index is the point ID.
    ids = np.arange(len(dataframe))

    try:
        groups = np.array([
            fold_map[int(i)]
            for i in ids
        ])
    except KeyError as e:
        raise ValueError(
            f"Point ID {e} from dataframe has no fold assignment."
        )

    return groups

In [14]:
def custom_cross_validation_2D_DSM_ML(region, dataframe, param_grid, normalization, model,
                                       folds, cv_method, soil_properties=soil_properties,
                                       depth='', plot_cv=True):
    """
    Grid-search + evaluate `model` for every soil property using a
    leave-one-fold-out scheme defined by `folds` (a [point_ID, fold_index, ...]
    list produced by randomCV.py / spatialCV.py, generated once in the cell
    above and shared by every model/normalization run).

    `cv_method` is a short label ('random', 'block', 'sp1', ...) used to
    keep results and saved models separate across CV strategies.
    """
    # Ensure output directories exist
    os.makedirs('../Results/evaluation_on_bestconfigs/', exist_ok=True)
    os.makedirs(f'../Results/evaluation_on_bestconfigs/{region}/{model}', exist_ok=True)
    os.makedirs(f'../Models/{region}/{model}/{cv_method}', exist_ok=True)
    os.makedirs(f'../Results/best_params/{region}/{model}/{cv_method}', exist_ok=True)
    
    # Create results CSV (one per model x CV method)
    experiments = f'experiments_{model}_{cv_method}_2d_dm_new_newest.csv'
    csv_columns = (['soil_property', 'cv_method'] + ['normalization', 'depth']
                   + list(param_grid.keys()) + ['rmse', 'mse', 'r2', 'mae', 'time'])
    df_experiments = create_results_csv(csv_columns, experiments)

    # Selecting features
    features = dataframe[covariates]

    # Map every row to its pre-computed fold (from randomCV / spatialCV)
    groups = build_group_array(dataframe, folds)
    unique_folds = np.sort(np.unique(groups))

    # Selecting scaler
    if normalization == 'MinMaxScaler':
        scaler = MinMaxScaler()
    elif normalization == 'StandardScaler':
        scaler = StandardScaler()
    else:
        scaler = None

    for s_property in soil_properties:
        y = dataframe[s_property]

        best_rmse = float('inf')
        best_r2 = -float('inf')
        best_model = None
        best_params = None

        for params in product(*param_grid.values()):
            param_dict = dict(zip(param_grid.keys(), params))
            print(param_dict)
            config_to_check = param_dict.copy()
            config_to_check['soil_property'] = s_property
            config_to_check['cv_method'] = cv_method
            config_to_check['normalization'] = normalization
            config_to_check['depth'] = depth
            exists = check_existance_params_in_experiments(df_experiments, config_to_check)
            if not exists:

                y_predict = []
                y_true = []
                start_time = time.time()

                for fold_val in unique_folds:
                    test_mask = groups == fold_val
                    train_mask = ~test_mask
                    train_index = np.where(train_mask)[0]
                    test_index = np.where(test_mask)[0]

                    X_train_all, X_test_all = features.iloc[train_index], features.iloc[test_index]
                    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
                    X_train = X_train_all[covariates]
                    X_test = X_test_all[covariates]

                    if scaler:
                        X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=covariates)
                        X_test = pd.DataFrame(scaler.transform(X_test), columns=covariates)
                    try:
                        if model == 'RandomForest':
                            training_model = random_forest_training(param_dict, X_train, y_train)
                        elif model == 'DecisionTree':
                            training_model = decision_tree_training(param_dict, X_train, y_train)
                        elif model == 'LGBM':
                            training_model = lgboost_training(param_dict, X_train, y_train, X_test, y_test)
                    except Exception as e:
                        print(f"An exception occurred: {e}")
                        raise

                    model_predict = training_model.predict(X_test)
                    y_predict.extend(model_predict)
                    y_true.extend(y_test.tolist())

                mse = mean_squared_error(y_true, y_predict)
                rmse = math.sqrt(mse)
                r2 = r2_score(y_true, y_predict)
                mae = mean_absolute_error(y_true, y_predict)
                training_time = time.time() - start_time

                if rmse < best_rmse and r2 > best_r2:
                    best_r2 = r2
                    best_rmse = rmse
                    best_model = training_model
                    best_params = param_dict

                row = param_dict.copy()
                row.update({
                    'soil_property': s_property,
                    'cv_method': cv_method,
                    'normalization': normalization,
                    'depth': depth,
                    'rmse': rmse,
                    'mse': mse,
                    'r2': r2,
                    'mae': mae,
                    'time': training_time,
                })

                df_experiments.loc[len(df_experiments)] = row
                df_experiments.to_csv(f"../Results/evaluation_on_bestconfigs/{region}/{model}/{experiments}", index=False)

        if best_model:
            dump(best_model, f'../Models/{region}/{model}/{cv_method}/best_{model}_{cv_method}_{depth}_{s_property}_{normalization}.joblib')
            best_params_df = pd.DataFrame([best_params])
            best_params_df['r2'] = best_r2
            best_params_df['rmse'] = best_rmse
            best_params_df.to_csv(
                f'../Results/best_params/{region}/{model}/{cv_method}/best_params_2D_{model}_{cv_method}_{depth}_{s_property}_{normalization}_newww.csv',
                index=False
            )

## Hyper parameters

In [15]:
param_grid_dt = {
   # 'ccp_alpha': [0, 0.1, 0.01, 0.001],
   'ccp_alpha': [0.1, 0.01, 0.001],
    'criterion': ['squared_error'],
    'max_depth': [None, 20],
    'min_samples_split': [2, 20],
    'min_samples_leaf': [1, 20],
    'max_features': ['log2', 0.8],
    'splitter': ['random'],
    'max_leaf_nodes': [None],
    'min_impurity_decrease': [0.0, 0.1],
}

param_grid_rf = {
    'max_features': [1.0, 'sqrt'],
    'ccp_alpha': [0.01, 0.1],
    'n_estimators': [100, 200],
    'min_impurity_decrease': [0.1, 0.0],
    'max_depth': [None],
    'bootstrap': [True, False],
    'criterion': ['squared_error', "absolute_error"],
    'random_state': [15],
  #  'min_samples_leaf': [1, 2, 3, 4, 20],
}

param_grid_lgbm = {
    'n_estimators': [200, 300],
    'min_split_gain': [0, 0.1],
   # 'num_leaves': [31],
    'max_depth': [-1, 50, 100],
    'learning_rate': [0.1, 0.01],
    'min_child_samples': [20, 50],
    'colsample_bytree': [0.6, 1.0],
    'reg_alpha': [0, 0.1],
    #'reg_lambda': [0, 0.1],
    # 'tree_learner': ['serial' , 'feature', 'data', 'voting'],
    # 'importance_type' : ['split', 'gain']
}


# param_grid_rf = {
#       'max_features': ['sqrt'], 'ccp_alpha': [0.01], 'n_estimators': [100],
#             'min_impurity_decrease': [0.0], 'max_depth': [None], 'bootstrap': [True],
#             'criterion': ['squared_error'], 'random_state': [15],
# }

# param_grid_lgbm = {
#   'n_estimators': [100], 'min_split_gain': [0], 'max_depth': [-1],
#          'learning_rate': [0.1], 'min_child_samples': [20], 'colsample_bytree': [1.0],
#          'reg_alpha': [0],
# }

# param_grid_dt = {
#    'ccp_alpha': [0.001], 'criterion': ['squared_error'], 'max_depth': [None],
#           'min_samples_split': [2], 'min_samples_leaf': [1], 'max_features': ['log2'],
#           'splitter': ['random'], 'max_leaf_nodes': [None], 'min_impurity_decrease': [0.0]
# }


all_params = {
    # 'DecisionTree': param_grid_dt,
    'RandomForest': param_grid_rf,
    'LGBM': param_grid_lgbm
}

normalizations = [
    # 'MinMaxScaler',
 #   'StandardScaler',
    "Without"
]


ML_models = [
   'RandomForest',
    'LGBM',
    # 'DecisionTree',
]

> A much smaller grid for a quick smoke-test of the pipeline (doesn't
> overwrite `all_params` above — assign it explicitly if you want to use it,
> e.g. `all_params = debug_params`).

In [16]:
debug_params = {
    'DecisionTree': {
        'ccp_alpha': [0.001], 'criterion': ['squared_error'], 'max_depth': [None],
        'min_samples_split': [2], 'min_samples_leaf': [1], 'max_features': ['log2'],
        'splitter': ['random'], 'max_leaf_nodes': [None], 'min_impurity_decrease': [0.0],
    },
    'RandomForest': {
        'max_features': ['sqrt'], 'ccp_alpha': [0.01], 'n_estimators': [100],
        'min_impurity_decrease': [0.0], 'max_depth': [None], 'bootstrap': [True],
        'criterion': ['squared_error'], 'random_state': [15],
    },
    'LGBM': {
        'n_estimators': [200], 'min_split_gain': [0], 'max_depth': [-1],
        'learning_rate': [0.1], 'min_child_samples': [20], 'colsample_bytree': [1.0],
        'reg_alpha': [0],
    },
}

In [17]:
# To train model with depths
# dataframe_2_5 = dataframe[dataframe['average profondeur'] == 2.5]
# dataframe_10 = dataframe[dataframe['average profondeur'] == 10]
# dataframe_22_5 = dataframe[dataframe['average profondeur'] == 22.5]

dfs = {
        # "2_5": dataframe_2_5,
        # "10": dataframe_10,
        # "22_5": dataframe_22_5,
        "all_depth": dataframe
        }

print(dfs.keys())

dict_keys(['all_depth'])


## Run experiments

Sweeps every model over every CV strategy generated above (`random`,
`block`, `sp1`), every depth subset, and every normalization. All runs for
a given CV method reuse the exact same fold assignment saved earlier, so
results are directly comparable across models within that method.

In [18]:
folds_by_method.keys()

dict_keys(['random', 'block_k5_5000', 'block_k5_10000', 'block_k5_20000', 'block_k5_30000', 'block_k5_50000', 'block_k5_100000', 'block_k5_150000', 'block_k5_200000', 'sp1_k5_5000', 'sp1_k5_10000', 'sp1_k5_20000', 'sp1_k5_30000', 'sp1_k5_50000', 'sp1_k5_100000', 'sp1_k5_150000', 'sp1_k5_200000', 'sp2_k5', 'sp_k5_5000', 'sp_k5_10000', 'sp_k5_20000', 'sp_k5_30000', 'sp_k5_50000', 'sp_k5_100000', 'sp_k5_150000', 'sp_k5_200000'])

In [19]:
folds_by_method.keys()

dict_keys(['random', 'block_k5_5000', 'block_k5_10000', 'block_k5_20000', 'block_k5_30000', 'block_k5_50000', 'block_k5_100000', 'block_k5_150000', 'block_k5_200000', 'sp1_k5_5000', 'sp1_k5_10000', 'sp1_k5_20000', 'sp1_k5_30000', 'sp1_k5_50000', 'sp1_k5_100000', 'sp1_k5_150000', 'sp1_k5_200000', 'sp2_k5', 'sp_k5_5000', 'sp_k5_10000', 'sp_k5_20000', 'sp_k5_30000', 'sp_k5_50000', 'sp_k5_100000', 'sp_k5_150000', 'sp_k5_200000'])

In [20]:
CV_METHODS = folds_by_method.keys()
CV_METHODS = ['sp2_k5' ]
# TODO: SELECT CROSS VALIDATION TECHNIQUES TO ADOPT 
for cv_method in CV_METHODS:
    folds = folds_by_method[cv_method]
    for model in ML_models:
        for depth, df in dfs.items():
            for normalization in normalizations:
                print(f"Training model={model} | CV={cv_method} | depth={depth} | normalization={normalization}")
                custom_cross_validation_2D_DSM_ML(
                    region = REGION, dataframe = df, param_grid = all_params[model], normalization=normalization, model=model,
                    folds=folds, cv_method=cv_method, depth=depth
                )

Training model=RandomForest | CV=sp2_k5 | depth=all_depth | normalization=Without
{'max_features': 1.0, 'ccp_alpha': 0.01, 'n_estimators': 100, 'min_impurity_decrease': 0.1, 'max_depth': None, 'bootstrap': True, 'criterion': 'squared_error', 'random_state': 15}
{'max_features': 1.0, 'ccp_alpha': 0.01, 'n_estimators': 100, 'min_impurity_decrease': 0.1, 'max_depth': None, 'bootstrap': True, 'criterion': 'absolute_error', 'random_state': 15}
{'max_features': 1.0, 'ccp_alpha': 0.01, 'n_estimators': 100, 'min_impurity_decrease': 0.1, 'max_depth': None, 'bootstrap': False, 'criterion': 'squared_error', 'random_state': 15}
{'max_features': 1.0, 'ccp_alpha': 0.01, 'n_estimators': 100, 'min_impurity_decrease': 0.1, 'max_depth': None, 'bootstrap': False, 'criterion': 'absolute_error', 'random_state': 15}
{'max_features': 1.0, 'ccp_alpha': 0.01, 'n_estimators': 100, 'min_impurity_decrease': 0.0, 'max_depth': None, 'bootstrap': True, 'criterion': 'squared_error', 'random_state': 15}
{'max_features

## Outputs

- `./plots/cv_folds/{random,block,sp1}/*.png`, `*.csv` — fold visualizations
  + `[ID, fold]` assignments for each CV strategy (generated once, reused by
  every model).
- `./results/experiments_{model}_{cv_method}_2d_dm_new.csv` — every
  hyperparameter combination tried, per model per CV method.
- `./models/{model}/{cv_method}/best_*.joblib` — best model per soil
  property / normalization, within each CV method.
- `./results/{model}/{cv_method}/best_params_2D_*.csv` — best hyperparameters
  + r2/rmse per soil property / normalization, within each CV method.